# Patient Journey - Procedures (NBA 3.1)

First dialysis date per patient from IQVIA LAAD procedure claims (`PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX`). `06_patient_journey_combine` turns it into `DIALYSIS_FLAG` per patient-month. The 3.0 query is kept untouched as a baseline (cell "Load px data"); each step adds a new query and a comparison with the previous one, the same approach as the lab and Dx notebooks.

* **Plan and decisions:** `Px Files/NBA_3_1_Px_Journey_Plan.md`
* **Px basket:** `Final Product Basket/Kerendia_Px_Final.csv` (one row per code, `PX_GROUP` and `ACTIVE_DIALYSIS` per code). The original file is kept unchanged as `Kerendia NBA 3.1 - PX Event List.csv`.
* **Old notebook (3.0 baseline):** `Px Files/(Clone) 05_patient_journey_px` (unchanged)
* **Output:** `kerendia_nba_3_1_px_validation` is written only after review (Step 5, `WRITE_OUTPUT = False`).

| Step | What changes |
| --- | --- |
| 0 | Nothing: the 3.0 query is checked against the table it last wrote |
| 1 | Nothing: hardcoded parts listed, and every code the 3.0 text match catches is profiled |
| 2 | Basket: the code list comes from the Px basket instead of the word `DIALYSIS` |
| 3 | Config: table and column names and the end date in `PX_CONFIG` |
| 4 | Code: first dialysis code and group in the output, dropped-rows report |
| 5 | Write (switched off until review) |

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../../00_cohort_config"

In [0]:


query = """
      SELECT DISTINCT PATIENT_ID,
                      min(SERVICE_DATE) as FIRST_DIALYSIS_DATE,    
                      1 as DIALYSIS_FLAG
                      --PX.PROCEDURE_CODE,
                      --PROCEDURE_DESCRIPTION
                      
      FROM (select PATIENT_ID, SERVICE_DATE, PROCEDURE_CODE from PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX)  PX
      JOIN 
            (   SELECT DISTINCT  PROCEDURE_CODE,PROCEDURE_DESCRIPTION
                FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROCEDURE 
                WHERE UPPER(PROCEDURE_DESCRIPTION) LIKE '%DIALYSIS%'
            ) PXCD
      ON PX.PROCEDURE_CODE = PXCD.PROCEDURE_CODE
      group by PATIENT_ID, DIALYSIS_FLAG
   """  
df_px = Get_Data_Snowflakes(query)
df_px.display()

In [0]:
# Step 0 check: the 3.0 query in this notebook must give the same output as the table last written by the old
# notebook ((Clone) 05_patient_journey_px, last write 2026-10-26). The query reads all history with no end date,
# so differences can only come from a data refresh; they are shown here, not hidden.
import pandas as pd
import pyspark.sql.functions as F

px_query_30 = query   # keep the 3.0 SQL under its own name (the name `query` is generic and could be reused)
PX_VALIDATION_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                      "hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_px_validation")

try:
    _old = spark.read.format("delta").load(PX_VALIDATION_PATH)
except Exception as e:
    _old = None
    print(f"Old validation table not readable, Step 0 check skipped: {str(e)[:200]}")

if _old is not None:
    _n = df_px.select(F.col("PATIENT_ID").cast("string").alias("PATIENT_ID"),
                      F.to_date("FIRST_DIALYSIS_DATE").alias("NEW"))
    _o = _old.select(F.col("PATIENT_ID").cast("string").alias("PATIENT_ID"),
                     F.to_date("FIRST_DIALYSIS_DATE").alias("OLD"), F.lit(1).alias("IN_OLD"))
    _j = _n.withColumn("IN_NEW", F.lit(1)).join(_o, ["PATIENT_ID"], "full")
    _s = _j.agg(
        F.count("*").alias("patients_union"),
        F.sum(F.when(F.col("IN_OLD").isNull(), 1).otherwise(0)).alias("only_this_notebook"),
        F.sum(F.when(F.col("IN_NEW").isNull(), 1).otherwise(0)).alias("only_old_table"),
        F.sum(F.when(F.col("NEW").eqNullSafe(F.col("OLD")) & F.col("IN_OLD").isNotNull() & F.col("IN_NEW").isNotNull(), 1).otherwise(0)).alias("same_first_date"),
        F.sum(F.when(~F.col("NEW").eqNullSafe(F.col("OLD")) & F.col("IN_OLD").isNotNull() & F.col("IN_NEW").isNotNull(), 1).otherwise(0)).alias("different_first_date"),
    ).toPandas()
    print(f"This notebook: {df_px.count():,} patients (rows); old table: {_old.count():,}")
    print(f"Rows with an empty PATIENT_ID: {df_px.filter(F.col('PATIENT_ID').isNull()).count():,} (3.0 groups them into one row)")
    display(_s)

### Step 1: manual (hardcoded) parts of the 3.0 Px query

Review of the cell "Load px data". Full detail in the plan file, section 2.

**What it does:** finds every procedure code whose description contains the word "DIALYSIS", and outputs one row per patient with the first date of any of those codes and `DIALYSIS_FLAG = 1`. All history, all US, no provider filter.

| # | Hardcoded today | Effect | Fixed in |
| --- | --- | --- | --- |
| Q1 | Which procedures count: description contains `'%DIALYSIS%'` | No list to review; any code with the word counts, any code without it is missed | Step 2 (basket) |
| Q2 | Wrong matches: C1751 catheter "(OTHER THAN HEMODIALYSIS)", 93985 / 93986 scans "prior to creation" of the access | First date too early, or a patient flagged who never had dialysis | Step 2 (basket) |
| Q3 | Possible misses: ESRD monthly services 90960-90962, home hemodialysis visit, access procedures without the word | Dialysis found late or not at all | Step 2 (basket) |
| Q4 | Mixed signals under one flag: sessions, ESRD monthly services, access (often created months before dialysis), training, drugs billed "for ESRD on dialysis" | One flag hides the difference | Step 2 (`PX_GROUP` per code) |
| Q5 | No end date | Future-dated rows can set the first date | Step 3 (`PX_CONFIG`) |
| Q6 | Table and column names typed in the SQL | A rename breaks the query | Step 3 (`PX_CONFIG`) |
| Q7 | `DISTINCT` + `GROUP BY` together; constant `DIALYSIS_FLAG = 1`; empty `PATIENT_ID` grouped into one row | Double work, a column with no information | Step 4 |
| Q8 | Code of the first event not in the output | Cannot check which code set the first date | Step 4 |
| Q9 | No dropped-rows report | Losses are silent | Step 4 |

**Not changed: the date window.** All history is kept on purpose (decision D3): the query looks for the *first* dialysis ever, and `06_patient_journey_combine` treats a first date before 2017 as "on dialysis". `DATA_START_DATE` would move first dates later and switch the flag off for long-term dialysis patients.

**Correction to the first review (2026-10-29):** drugs billed "for ESRD on dialysis" (epoetin Q4081 / J0887 / Q5105, darbepoetin J0882, cinacalcet J0604, difelikefalin J0879) are *not* wrong matches: they are billed only for patients on dialysis, so they count (decision D1).

The cell below (1b) profiles every code the 3.0 text match catches, so the effect of each decision is visible before Step 2.

In [0]:
# 1b. Every procedure code the 3.0 text match ('%DIALYSIS%' in the description) catches, all history, all US:
# rows, patients, first / last date seen, and for how many patients the code sets FIRST_DIALYSIS_DATE in 3.0.
# The table names here are the 3.0 ones; they move to PX_CONFIG in Step 3.
PX_FACT_30 = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX"
PX_DIM_30 = "PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROCEDURE"
TEXT_MATCH_CODES_SQL = (f"SELECT DISTINCT PROCEDURE_CODE FROM {PX_DIM_30} "
                        f"WHERE UPPER(PROCEDURE_DESCRIPTION) LIKE '%DIALYSIS%'")

def first_events_sql(code_filter_sql, fact=PX_FACT_30):
    # first date per patient for a set of codes, and the code on that date (ties: lowest code)
    return f"""
SELECT PATIENT_ID, SERVICE_DATE AS FIRST_DATE, PROCEDURE_CODE AS FIRST_CODE
FROM {fact}
WHERE PROCEDURE_CODE IN ({code_filter_sql}) AND SERVICE_DATE IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY SERVICE_DATE, PROCEDURE_CODE) = 1"""

px_profile = Get_Data_Snowflakes(f"""
WITH D AS (SELECT PROCEDURE_CODE, MAX(PROCEDURE_DESCRIPTION) AS PROCEDURE_DESCRIPTION
           FROM {PX_DIM_30} WHERE UPPER(PROCEDURE_DESCRIPTION) LIKE '%DIALYSIS%' GROUP BY 1),
     S AS (SELECT P.PROCEDURE_CODE, COUNT(*) AS ROWS_, COUNT(DISTINCT P.PATIENT_ID) AS PATIENTS,
                  MIN(P.SERVICE_DATE) AS FIRST_SEEN, MAX(P.SERVICE_DATE) AS LAST_SEEN
           FROM {PX_FACT_30} P JOIN D ON P.PROCEDURE_CODE = D.PROCEDURE_CODE GROUP BY 1),
     F AS ({first_events_sql(TEXT_MATCH_CODES_SQL)}),
     FC AS (SELECT FIRST_CODE, COUNT(*) AS PATIENTS_FIRST_DATE FROM F GROUP BY 1)
SELECT D.PROCEDURE_CODE, D.PROCEDURE_DESCRIPTION, COALESCE(S.ROWS_, 0) AS ROWS_, COALESCE(S.PATIENTS, 0) AS PATIENTS,
       S.FIRST_SEEN, S.LAST_SEEN, COALESCE(FC.PATIENTS_FIRST_DATE, 0) AS PATIENTS_FIRST_DATE
FROM D LEFT JOIN S ON D.PROCEDURE_CODE = S.PROCEDURE_CODE
LEFT JOIN FC ON D.PROCEDURE_CODE = FC.FIRST_CODE
ORDER BY PATIENTS DESC""").toPandas()
for _c in ["ROWS_", "PATIENTS", "PATIENTS_FIRST_DATE"]:
    px_profile[_c] = px_profile[_c].astype("int64")

print(f"Codes with 'DIALYSIS' in the description: {len(px_profile):,}; with claims: {(px_profile.PATIENTS > 0).sum():,}")
print(f"Patients with a first date (3.0 logic): {px_profile.PATIENTS_FIRST_DATE.sum():,}")
display(px_profile)

## Step 2: Basket changes (Q1-Q4)

The code list now comes from `Final Product Basket/Kerendia_Px_Final.csv`: only codes with `ACTIVE_DIALYSIS = Y` count (decision D1: dialysis sessions, ESRD monthly services, home hemodialysis visit, dialysis training, drugs billed "for ESRD on dialysis"). Access procedures, pre-access scans, transplant and C1751 are in the basket with their `PX_GROUP` but do not count. Everything else (table, all history, output columns) is kept as in 3.0, so the comparison shows only the effect of the basket.

| Cell | What it does |
| --- | --- |
| 2a | Reads the basket, expands clubbed rows, checks it, checks it has the same codes as the original Px Event List, and cross-checks it with the 1b profile (decision D2) |
| 2b | **The Px query** (built from the basket) |
| 2c | Runs it |
| 2d | Compares with 3.0: all US patients (first date) and validation patients (dialysis flag per month, as `06_patient_journey_combine` builds it) |
| 2e | Findings |

In [0]:
# 2a. Read the Px basket and check it is valid before building the query.
#
# Basket file: one row per code (MATCH_TYPE = CODE) or per clubbed group (MATCH_TYPE = LIST, member codes separated by |)
#   CONDITION        CKD / HF / DM / T1D / LIPIDS / OTHER     (what the code is about)
#   PX_GROUP         DIALYSIS_SESSION / ESRD_MONTHLY / DIALYSIS_TRAINING / DIALYSIS_DRUG / DIALYSIS_ACCESS / PRE_ACCESS / TRANSPLANT / empty
#   ACTIVE_DIALYSIS  Y = counts for FIRST_DIALYSIS_DATE (decision D1)
# Path: our copy of the basket folder (the Dx notebook reads wenjie.chen's _sm copy, which does not have this file yet).
_BASKET_DIR = ("/Workspace/Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_develop_hcp_claims/NBA 3.1 - Dev/"
               "03_Feature_Engineering/03_HCP_level_features/Final Product Basket")
PX_BASKET_PATH = f"{_BASKET_DIR}/Kerendia_Px_Final.csv"
PX_EVENT_LIST_PATH = f"{_BASKET_DIR}/Kerendia NBA 3.1 - PX Event List.csv"   # original list, only used for the check below

PX_GROUPS = {"", "DIALYSIS_SESSION", "ESRD_MONTHLY", "DIALYSIS_TRAINING", "DIALYSIS_DRUG",
             "DIALYSIS_ACCESS", "PRE_ACCESS", "TRANSPLANT"}
PX_GROUPS_DIALYSIS = {"DIALYSIS_SESSION", "ESRD_MONTHLY", "DIALYSIS_TRAINING", "DIALYSIS_DRUG"}   # decision D1

px_basket_raw = pd.read_csv(PX_BASKET_PATH, dtype=str).fillna("")
px_basket = (px_basket_raw.assign(PROCEDURE_CODE=px_basket_raw["MATCH_VALUE"].str.split("|"))
             .explode("PROCEDURE_CODE").reset_index(drop=True))
px_basket["PROCEDURE_CODE"] = px_basket["PROCEDURE_CODE"].str.strip()

# Checks: the cell stops with a clear message if the file is not valid
_raw = px_basket_raw
assert set(_raw["MATCH_TYPE"]) <= {"CODE", "LIST"}, "MATCH_TYPE must be CODE or LIST"
assert (~_raw.loc[_raw.MATCH_TYPE == "CODE", "MATCH_VALUE"].str.contains(r"\|")).all(), "A CODE row has several codes"
assert (_raw.loc[_raw.MATCH_TYPE == "LIST", "CLUB_ID"] != "").all(), "A LIST row has no CLUB_ID"
assert not px_basket["PROCEDURE_CODE"].duplicated().any(), \
    f"Codes listed twice: {px_basket.loc[px_basket.PROCEDURE_CODE.duplicated(), 'PROCEDURE_CODE'].tolist()}"
assert set(px_basket["ACTIVE_DIALYSIS"]) <= {"Y", "N"}, "ACTIVE_DIALYSIS must be Y or N"
assert set(px_basket["PX_GROUP"]) <= PX_GROUPS, f"Unknown PX_GROUP: {set(px_basket['PX_GROUP']) - PX_GROUPS}"
assert set(px_basket["CONDITION"]) <= {"CKD", "HF", "DM", "T1D", "LIPIDS", "OTHER"}, "Unknown CONDITION"
assert px_basket.loc[px_basket.ACTIVE_DIALYSIS == "Y", "PX_GROUP"].isin(PX_GROUPS_DIALYSIS).all(), \
    "ACTIVE_DIALYSIS = Y only on the dialysis groups of decision D1"
assert set(px_basket["IS_DEFINING"]) <= {"Y", "N", "MIXED"} and set(px_basket["IN_752_FEATURES"]) <= {"Y", "N"}, \
    "IS_DEFINING must be Y / N / MIXED and IN_752_FEATURES Y / N"

# The basket must have exactly the codes of the original Px Event List (nothing lost or added in the rebuild),
# with the same patient counts and flags.
# Exception: rows whose NOTE starts with "Added" are new on purpose (decision D2, 2026-10-29: codes caught by the
# 3.0 text match that were not in the original list). They are left out of this check and listed below instead.
_added = _raw[_raw["NOTE"].str.startswith("Added")]
_added_codes = {c for v in _added["MATCH_VALUE"] for c in v.split("|")}
_orig = pd.read_csv(PX_EVENT_LIST_PATH, dtype=str).fillna("")
_club = _orig["is_clubbed"].str.upper() == "YES"
_members = lambda s: frozenset(m.strip() for m in s.replace("|", ",").split(",") if m.strip())
_orig_codes = set(_orig.loc[~_club, "PROCEDURE_CODE"]) | {m for x in _orig.loc[_club, "club_members"] for m in _members(x)}
_missing, _extra = _orig_codes - set(px_basket.PROCEDURE_CODE), set(px_basket.PROCEDURE_CODE) - _orig_codes - _added_codes
assert not _missing and not _extra, f"Codes differ from the original list. Missing: {sorted(_missing)}; extra: {sorted(_extra)}"
_yn = lambda s: s.str.upper().map({"YES": "Y", "NO": "N", "MIXED": "MIXED"})
_o = _orig.assign(KEY=[_members(m) if c else frozenset([p]) for p, m, c in zip(_orig.PROCEDURE_CODE, _orig.club_members, _club)],
                  DEF=_yn(_orig.is_defining), F752=_yn(_orig.in_752_features))
_b = _raw.drop(_added.index).assign(KEY=lambda d: d.MATCH_VALUE.map(_members))
_cmp = _b.merge(_o, on="KEY", how="outer", indicator=True)
_bad = _cmp[(_cmp._merge != "both") | (_cmp.PATIENTS_TOTAL != _cmp.patients_total)
            | (_cmp.IS_DEFINING != _cmp.DEF) | (_cmp.IN_752_FEATURES != _cmp.F752)]
assert _bad.empty, f"Rows differ from the original list: {_bad.MATCH_VALUE.tolist()}"

_active = px_basket[px_basket["ACTIVE_DIALYSIS"] == "Y"]
print(f"Basket OK: {len(_raw)} rows ({(_raw.MATCH_TYPE == 'LIST').sum()} clubbed) = {len(px_basket)} codes; "
      f"{len(_raw) - len(_added)} rows have the same codes, patient counts and flags as the original Px Event List "
      f"({len(_orig)} rows); {len(_added)} rows added (D2)")
if len(_added):
    display(_added[["MATCH_VALUE", "PROCEDURE_DESCRIPTION", "PX_GROUP", "ACTIVE_DIALYSIS", "NOTE"]])
print(f"Counted as dialysis (ACTIVE_DIALYSIS = Y): {len(_active)} codes")
display(px_basket[px_basket.PX_GROUP != ""].groupby(["PX_GROUP", "ACTIVE_DIALYSIS"])
        .agg(codes=("PROCEDURE_CODE", lambda s: ", ".join(s)), n=("PROCEDURE_CODE", "size")).reset_index())

# Cross-check with the 1b profile (decision D2: codes caught by the text match must not be lost silently)
_tm = px_profile[px_profile.PATIENTS > 0]
_in_b = set(px_basket.PROCEDURE_CODE)
_y = set(_active.PROCEDURE_CODE)
def _show(pdf):
    # display() cannot show an empty pandas table; print a line instead
    display(pdf) if len(pdf) else print("   (none)")
print("A. Caught by the 3.0 text match (with claims) but NOT in the basket -> to add with a PX_GROUP (decision D2):")
_show(_tm[~_tm.PROCEDURE_CODE.isin(_in_b)])
print("B. Caught by the 3.0 text match, in the basket, but NOT counted (ACTIVE_DIALYSIS = N) -> removed on purpose (D1):")
_show(_tm[_tm.PROCEDURE_CODE.isin(_in_b - _y)].merge(px_basket[["PROCEDURE_CODE", "PX_GROUP"]], on="PROCEDURE_CODE"))
print("C. Counted in the basket but NOT caught by the 3.0 text match -> new in Step 2:")
_show(_active[~_active.PROCEDURE_CODE.isin(set(px_profile.PROCEDURE_CODE))][["PROCEDURE_CODE", "PROCEDURE_DESCRIPTION", "PX_GROUP", "PATIENTS_TOTAL"]])

In [0]:
# 2b. The Px query, built from the basket (2a). Same table, same history (all) and same output as the 3.0 query;
# only the code list changes: codes with ACTIVE_DIALYSIS = Y instead of the word DIALYSIS in the description.
# Table names and the end date are still as in 3.0 (Step 3).

def basket_codes_sql(basket):
    codes = sorted(basket.loc[basket["ACTIVE_DIALYSIS"] == "Y", "PROCEDURE_CODE"])
    return ", ".join(f"'{c}'" for c in codes)

def build_px_query_basket(basket):
    return f"""
SELECT PATIENT_ID,
       MIN(SERVICE_DATE) AS FIRST_DIALYSIS_DATE,
       1 AS DIALYSIS_FLAG
FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX
WHERE PROCEDURE_CODE IN ({basket_codes_sql(basket)})
GROUP BY PATIENT_ID
"""

px_query_basket = build_px_query_basket(px_basket)
print(px_query_basket)

In [0]:
# 2c. Run the basket query (all US, as 3.0)
df_px_basket = Get_Data_Snowflakes(px_query_basket)
print(f"Basket query: {df_px_basket.count():,} patients (3.0 query: {df_px.count():,})")

In [0]:
# 2d. Compare the 3.0 query with the basket query. The helpers are reused in 3d and 4c.
# Part A: all US patients, in Snowflake: same first date / earlier / later / gained / lost.
# Part B: the same split by the code that caused it (gained and earlier: new first code; lost and later: 3.0 first code).
# Part C: validation patients (patients in the lab / Dx / Rx validation tables, i.e. of the 10 validation HCPs),
#         DIALYSIS_FLAG per patient-month built exactly as in 06_patient_journey_combine.

def compare_first_dates(q_old, q_new):
    return Get_Data_Snowflakes(f"""
WITH O AS (SELECT PATIENT_ID, FIRST_DIALYSIS_DATE AS D FROM ({q_old})),
     N AS (SELECT PATIENT_ID, FIRST_DIALYSIS_DATE AS D FROM ({q_new}))
SELECT CASE WHEN O.PATIENT_ID IS NULL AND N.PATIENT_ID IS NULL THEN '0. empty PATIENT_ID'
            WHEN O.PATIENT_ID IS NULL THEN '4. gained (only new)'
            WHEN N.PATIENT_ID IS NULL THEN '5. lost (only old)'
            WHEN O.D = N.D THEN '1. same first date'
            WHEN N.D < O.D THEN '2. new first date earlier'
            ELSE '3. new first date later' END AS RESULT,
       COUNT(*) AS PATIENTS,
       MEDIAN(DATEDIFF('day', O.D, N.D)) AS MEDIAN_DAYS_SHIFT
FROM O FULL OUTER JOIN N ON O.PATIENT_ID = N.PATIENT_ID
GROUP BY 1 ORDER BY 1""").toPandas()

def compare_by_code(old_codes_sql, new_codes_sql, top=15):
    _r = Get_Data_Snowflakes(f"""
WITH O AS ({first_events_sql(old_codes_sql)}), N AS ({first_events_sql(new_codes_sql)}),
     J AS (SELECT CASE WHEN O.PATIENT_ID IS NULL THEN '4. gained' WHEN N.PATIENT_ID IS NULL THEN '5. lost'
                       WHEN O.FIRST_DATE = N.FIRST_DATE THEN '1. same' WHEN N.FIRST_DATE < O.FIRST_DATE THEN '2. earlier'
                       ELSE '3. later' END AS RESULT,
                  CASE WHEN O.PATIENT_ID IS NULL OR N.FIRST_DATE < O.FIRST_DATE THEN N.FIRST_CODE ELSE O.FIRST_CODE END AS CODE
           FROM O FULL OUTER JOIN N ON O.PATIENT_ID = N.PATIENT_ID)
SELECT RESULT, CODE AS PROCEDURE_CODE, COUNT(*) AS PATIENTS FROM J WHERE RESULT <> '1. same' GROUP BY 1, 2""").toPandas()
    _r["PATIENTS"] = _r["PATIENTS"].astype("int64")
    _d = pd.concat([px_basket[["PROCEDURE_CODE", "PROCEDURE_DESCRIPTION", "PX_GROUP", "ACTIVE_DIALYSIS"]],
                    px_profile.loc[~px_profile.PROCEDURE_CODE.isin(px_basket.PROCEDURE_CODE), ["PROCEDURE_CODE", "PROCEDURE_DESCRIPTION"]]])
    _r = _r.merge(_d, on="PROCEDURE_CODE", how="left").sort_values(["RESULT", "PATIENTS"], ascending=[True, False])
    return _r.groupby("RESULT").head(top)

# validation spine: the patient-months of the lab / Dx / Rx validation tables (what 06_patient_journey_combine joins)
_QO = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
       "hcp_level_model/ops_pipeline/query_output/")
_spines = []
for _t in ["kerendia_nba_3_1_lab_journey_validation", "kerendia_nba_3_1_dx_journey_validation", "kerendia_nba_3_1_rx_journey_validation"]:
    try:
        _spines.append(spark.read.format("delta").load(_QO + _t)
                       .select(F.col("PATIENT_ID").cast("string").alias("PATIENT_ID"), F.to_date("DATE").alias("DATE")))
    except Exception as e:
        print(f"{_t} not readable, skipped: {str(e)[:120]}")
val_spine = _spines[0]
for _s in _spines[1:]:
    val_spine = val_spine.unionByName(_s)
val_spine = val_spine.distinct().cache()
val_patients = val_spine.select("PATIENT_ID").distinct()
print(f"Validation spine: {val_spine.count():,} patient-months, {val_patients.count():,} patients")

def pj06_dialysis_flag(px_sdf):
    # same rule as 06_patient_journey_combine, cell "Add dialysis flag"
    _p = (px_sdf.select(F.col("PATIENT_ID").cast("string").alias("PATIENT_ID"), "FIRST_DIALYSIS_DATE")
          .join(val_patients, "PATIENT_ID"))
    return (val_spine.join(_p, "PATIENT_ID", "left")
            .withColumn("FLAG", F.when(F.col("FIRST_DIALYSIS_DATE") < "2017-01-01", 1).otherwise(
                F.when(F.trunc(F.to_date("FIRST_DIALYSIS_DATE"), "month") <= F.col("DATE"), 1).otherwise(0)))
            .na.fill(0, subset=["FLAG"]).select("PATIENT_ID", "DATE", "FLAG"))

def compare_validation(px_old, px_new):
    _j = (pj06_dialysis_flag(px_old).withColumnRenamed("FLAG", "OLD")
          .join(pj06_dialysis_flag(px_new).withColumnRenamed("FLAG", "NEW"), ["PATIENT_ID", "DATE"]))
    _m = _j.agg(F.count("*").alias("patient_months"),
                F.sum("OLD").alias("months_flag_1_old"), F.sum("NEW").alias("months_flag_1_new"),
                F.sum(F.when(F.col("OLD") == F.col("NEW"), 1).otherwise(0)).alias("months_same"),
                F.sum(F.when((F.col("OLD") == 0) & (F.col("NEW") == 1), 1).otherwise(0)).alias("months_0_to_1"),
                F.sum(F.when((F.col("OLD") == 1) & (F.col("NEW") == 0), 1).otherwise(0)).alias("months_1_to_0")).toPandas()
    _p = (_j.groupBy("PATIENT_ID").agg(F.max("OLD").alias("OLD"), F.max("NEW").alias("NEW"))
          .agg(F.sum("OLD").alias("patients_ever_flagged_old"), F.sum("NEW").alias("patients_ever_flagged_new"))).toPandas()
    return pd.concat([_m, _p], axis=1)

print("Part A - all US patients, first dialysis date: 3.0 vs basket")
display(compare_first_dates(px_query_30, px_query_basket))
print("Part B - by the code that caused the difference (top 15 per result)")
display(compare_by_code(TEXT_MATCH_CODES_SQL, basket_codes_sql(px_basket)))
print("Part C - validation patients, DIALYSIS_FLAG per patient-month (06_patient_journey_combine rule)")
display(compare_validation(df_px, df_px_basket))

### 2e. Findings: basket query vs 3.0 query (run 2026-10-29)

**Basket (2a):** 145 rows = 152 codes. The 130 original rows match the Px Event List exactly. 15 rows were added under decision D2: the codes the 3.0 text match catches that were not in the list (7 ESRD services, 4 ESRD drugs counted; 4 access codes not counted). After the additions, list A is empty, so no 3.0 code is lost silently. **29 codes count as dialysis** (sessions 6, ESRD monthly 11, training 2, drugs 10).

**Part A, all US patients (first dialysis date):**

| Result | Patients | Median shift |
| --- | --- | --- |
| Same first date | 1,087,148 | 0 days |
| New first date earlier | 268,617 | 134 days earlier |
| New first date later | 45,596 | 20 days later |
| Gained (only in the basket query) | 134,972 | |
| Lost (only in 3.0) | 87,841 | |
| **Total** | 1,489,202 → **1,536,333** (+47,131, +3.2%) | |

**Part B, by the code that caused it:**
* **Earlier and gained come almost entirely from the ESRD monthly services 90960 / 90961 / 90962.** Their description has no word "DIALYSIS", so 3.0 missed them. They account for 268,377 of the 268,617 earlier patients and 134,673 of the 134,972 gained. The rest is home hemodialysis visits (99512) and difelikefalin (J0879).
* **Later and lost come from access codes that no longer count (D1):** long-term catheter C1750, cannula 36800 and short-term catheter C1752. They account for 45,237 of the 45,596 later patients and 87,319 of the 87,841 lost. These 87,841 patients have only an access procedure and no dialysis session, ESRD service or ESRD drug in the data.

**Part C, validation patients (4,238 patients, 137,373 patient-months, `DIALYSIS_FLAG` as built in `06_patient_journey_combine`):**

| | 3.0 | Basket |
| --- | --- | --- |
| Patient-months with `DIALYSIS_FLAG = 1` | 7,649 | 8,413 |
| Patients ever flagged | 325 | 363 |
| Months 0 → 1 / 1 → 0 | | 844 / 80 |

**Observations**
* The text match had **no wrong matches** of the kind first expected. C1751 and 93985 / 93986 are not caught by it. The "too early" first dates came from **access** codes (catheters, cannula), which are often placed before dialysis starts.
* **Procedure history starts on 2021-01-02** (first date seen for every code). "First dialysis ever" therefore means "first since 2021", and the `< 2017-01-01` rule in `06_patient_journey_combine` never applies.
* **For review:** the 87,841 access-only patients are no longer flagged. If the team prefers to count access as dialysis, set `ACTIVE_DIALYSIS = Y` on the `DIALYSIS_ACCESS` rows of the basket. No code change is needed.

In [0]:
# 2f. Check for the access decision (D1): do the patients lost in Step 2 (access procedure only) have a diagnosis
# of end-stage disease or dialysis? If yes, the Dx journey already marks them as stage 5 / not eligible.
# Groups: the 3.0 first code of each lost patient; reference = patients still counted in 3.1.
# Dx codes are looked for at any time in the Dx data (before or after the access procedure).
DX_FACT = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX"

access_dx_check = Get_Data_Snowflakes(f"""
WITH F AS ({first_events_sql(TEXT_MATCH_CODES_SQL)}),
     NEW AS (SELECT DISTINCT PATIENT_ID FROM ({px_query_basket})),
     P AS (SELECT F.PATIENT_ID,
                  CASE WHEN N.PATIENT_ID IS NOT NULL THEN '0. Counted in 3.1 (reference)'
                       WHEN F.FIRST_CODE = 'C1752' THEN '1. Lost: short-term catheter C1752'
                       WHEN F.FIRST_CODE = 'C1750' THEN '2. Lost: long-term catheter C1750'
                       WHEN F.FIRST_CODE = '36800' THEN '3. Lost: cannula 36800'
                       ELSE '4. Lost: other access' END AS GRP
           FROM F LEFT JOIN NEW N ON F.PATIENT_ID = N.PATIENT_ID),
     D AS (SELECT PATIENT_ID,
                  MAX(IFF(DIAGNOSIS_CODE = 'N18.6', 1, 0)) AS ESRD,
                  MAX(IFF(DIAGNOSIS_CODE = 'Z99.2', 1, 0)) AS DIAL_DEP,
                  MAX(IFF(DIAGNOSIS_CODE LIKE 'Z49.%', 1, 0)) AS DIAL_ENC,
                  MAX(IFF(DIAGNOSIS_CODE IN ('N18.5', 'I12.0', 'I13.11', 'I13.2'), 1, 0)) AS STAGE5,
                  MAX(IFF(DIAGNOSIS_CODE = 'N18.4', 1, 0)) AS STAGE4,
                  MAX(IFF(DIAGNOSIS_CODE LIKE 'N17%', 1, 0)) AS AKI,
                  MAX(IFF(DIAGNOSIS_CODE IN ('Z94.0', 'Z48.22'), 1, 0)) AS TRANSPLANT
           FROM {DX_FACT}
           WHERE (DIAGNOSIS_CODE IN ('N18.6', 'Z99.2', 'N18.5', 'I12.0', 'I13.11', 'I13.2', 'N18.4', 'Z94.0', 'Z48.22')
                  OR DIAGNOSIS_CODE LIKE 'Z49.%' OR DIAGNOSIS_CODE LIKE 'N17%')
             AND PATIENT_ID IN (SELECT PATIENT_ID FROM F)
           GROUP BY 1)
SELECT P.GRP, COUNT(*) AS PATIENTS,
       ROUND(100 * AVG(COALESCE(GREATEST(D.ESRD, D.DIAL_DEP, D.DIAL_ENC), 0)), 1) AS PCT_ESRD_OR_DIALYSIS_DX,
       ROUND(100 * AVG(COALESCE(GREATEST(D.ESRD, D.DIAL_DEP, D.DIAL_ENC, D.STAGE5), 0)), 1) AS PCT_STAGE5_ESRD_OR_DIALYSIS,
       ROUND(100 * AVG(COALESCE(D.ESRD, 0)), 1) AS PCT_N18_6,
       ROUND(100 * AVG(COALESCE(D.DIAL_DEP, 0)), 1) AS PCT_Z99_2,
       ROUND(100 * AVG(COALESCE(D.DIAL_ENC, 0)), 1) AS PCT_Z49,
       ROUND(100 * AVG(COALESCE(D.STAGE4, 0)), 1) AS PCT_STAGE4,
       ROUND(100 * AVG(COALESCE(D.AKI, 0)), 1) AS PCT_AKI_N17,
       ROUND(100 * AVG(COALESCE(D.TRANSPLANT, 0)), 1) AS PCT_TRANSPLANT,
       ROUND(100 * AVG(IFF(D.PATIENT_ID IS NULL, 1, 0)), 1) AS PCT_NONE_OF_THESE
FROM P LEFT JOIN D ON P.PATIENT_ID = D.PATIENT_ID
GROUP BY 1 ORDER BY 1""").toPandas()

access_dx_check["PATIENTS"] = access_dx_check["PATIENTS"].astype("int64")
print(f"Lost patients checked: {access_dx_check.loc[access_dx_check.GRP.str.contains('Lost'), 'PATIENTS'].sum():,} (expected 87,841)")
display(access_dx_check)

# Code-format check: how the kidney codes are written in the Dx fact (with / without dot), one recent month
print("Code format in the Dx fact (Sep 2026): kidney failure codes N17 / N18 / N19")
display(Get_Data_Snowflakes(f"""
SELECT DIAGNOSIS_CODE, COUNT(*) AS ROWS_ FROM {DX_FACT}
WHERE SERVICE_DATE >= '2026-09-01' AND (DIAGNOSIS_CODE LIKE 'N17%' OR DIAGNOSIS_CODE LIKE 'N18%' OR DIAGNOSIS_CODE LIKE 'N19%')
GROUP BY 1 ORDER BY 2 DESC LIMIT 20"""))

### 2g. Findings: access-only patients in the Dx data (run 2026-10-29)

Question: the 87,841 patients lost in Step 2 had only an access procedure. Are they really on dialysis, so that we should keep flagging them? Check: do they have an end-stage or dialysis diagnosis (`N18.6`, `Z99.2`, `Z49.x`) or a stage-5 code anywhere in the Dx claims? Patients still counted in 3.1 are the reference.

| Group (3.0 first code) | Patients | ESRD or dialysis Dx | + stage 5 | N18.6 | Z99.2 | Stage 4 | Transplant | None of these |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Reference: counted in 3.1 | 1,401,361 | 88.6% | 89.2% | 84.2% | 83.4% | 37.9% | 5.0% | 9.3% |
| Short-term catheter C1752 | 24,300 | 8.8% | 9.6% | 5.6% | 4.8% | 6.4% | 0.4% | **87.5%** |
| Long-term catheter C1750 | 31,669 | **48.6%** | 50.2% | 43.2% | 39.0% | 28.2% | 3.4% | 47.7% |
| Cannula 36800 | 31,350 | 25.7% | 28.0% | 19.2% | 13.0% | 14.3% | 0.6% | 65.8% |
| Other access | 522 | 57.1% | 57.5% | 55.9% | 46.7% | 22.2% | 5.9% | 41.0% |

The lost patients add up to 87,841, as in 2d. The Dx codes are written with a dot (`N18.6`), and the reference group shows that the check works: 9 in 10 patients counted as on dialysis also have a dialysis diagnosis.

**What it means**
* **Short-term catheter C1752: almost certainly not chronic dialysis.** 87.5% have no kidney failure, dialysis or even stage-4 diagnosis. A short-term catheter is used for acute, temporary dialysis (for example in hospital, after acute kidney injury) and for other procedures such as plasma exchange. Acute kidney injury (`N17`) cannot be checked: the Dx fact table has no `N17` (or `N19`) rows: of the kidney codes, only `N18.x` are in it (format check at the end of 2f), so the AKI column is 0 for every group. The table seems limited to a selected code list; AKI would need another source.
* **Long-term catheter C1750: about half are on dialysis.** 48.6% have an ESRD or dialysis diagnosis; 43.2% have `N18.6`, which gives them stage 5 in the Dx journey. The other half have no such diagnosis: catheters placed in preparation, for other uses, or dialysis billed outside these claims.
* **Cannula 36800: about one in four on dialysis.** Most are access creation before dialysis, or never followed by dialysis.
* Across all 87,841: about 25,900 (29%) have an ESRD or dialysis diagnosis; about 62,000 (71%) have none, and about 57,200 (65%) have none of the codes checked, not even stage 4.

**Decision D1 (confirmed 2026-10-29): access procedures stay out of the dialysis flag. The 3.1 Px table goes live only together with the 3.1 Dx table (Step 5 stays off until then).** Reasons:
* In `06_patient_journey_combine`, `DIALYSIS_FLAG = 1` sets `ELIGIBLE_PATIENT_FLAG = 0` from that month on. Counting access would make about 62,000 patients without any dialysis diagnosis permanently not eligible, almost all the C1752 patients among them.
* The patients who really are in end stage are already caught by the Dx journey: `N18.6` / stage-5 codes give stage 5, which is not eligible in PJ06. This is true for 43% of the C1750 group.
* **Condition:** the stage-5 safety net only exists in the **3.1 Dx** output. `06_patient_journey_combine` still reads the 3.0 Dx table, where `N18.6` gives no stage. So the 3.1 Px table should be written **together with** the 3.1 Dx table, not before; otherwise some of the about 26,000 patients with an ESRD or dialysis diagnosis would become eligible for a while.
* Not recommended: counting only C1750 as dialysis. Half of those patients have no dialysis diagnosis, and the other half are covered by Dx anyway.
* Access stays in the basket (`PX_GROUP = DIALYSIS_ACCESS`) for a Phase B feature such as "first access date" (a patient close to dialysis).

## Step 3: Config changes (Q5, Q6)

Run settings move into one block, `PX_CONFIG`: source table and column names, the end date (`date_end` from `00_config`, applied on `SERVICE_DATE` in the SQL) and the start date, which stays empty on purpose (decision D3: all history, the first dialysis ever is needed). The basket logic is the same as Step 2.

| Cell | What it does |
| --- | --- |
| 3a | `PX_CONFIG` and checks that tables and columns exist |
| 3b | **The Px query** (basket + config) |
| 3c | Runs it |
| 3d | Compares with Step 2 |
| 3e | Findings |

In [0]:
# 3a. Run settings for the Px query. Change values here, not in the SQL.
PX_CONFIG = {
    "start_date": None,                       # decision D3: all history (first dialysis ever; PJ06 treats < 2017 as on dialysis)
    "end_date": str(date_end)[:10],           # from 00_config (3.0 had no end date)
    "source": {
        "table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX",
        "columns": {"patient_id": "PATIENT_ID", "service_date": "SERVICE_DATE", "procedure_code": "PROCEDURE_CODE"},
    },
    # procedure dimension: only used for QC (descriptions); the query no longer needs it
    "procedure_dim": {
        "table": "PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROCEDURE",
        "columns": {"procedure_code": "PROCEDURE_CODE", "description": "PROCEDURE_DESCRIPTION"},
    },
}

# Checks: tables exist and every mapped column is there
for _blk in ("source", "procedure_dim"):
    _t = PX_CONFIG[_blk]["table"]
    _cols = set(c.upper() for c in Get_Data_Snowflakes(f"SELECT * FROM {_t} LIMIT 0").columns)
    _miss = [c for c in PX_CONFIG[_blk]["columns"].values() if c.upper() not in _cols]
    assert not _miss, f"{_t}: columns not found {_miss}"
assert PX_CONFIG["start_date"] is None or PX_CONFIG["start_date"] <= PX_CONFIG["end_date"], "start_date is after end_date"
print(f"PX_CONFIG OK: {PX_CONFIG['start_date'] or 'all history'} -> {PX_CONFIG['end_date']}, tables and columns found")

In [0]:
# 3b. The Px query from the basket (2a) and PX_CONFIG (3a). Output as in Step 2.

def _px_where(basket, cfg, alias="A"):
    sc = cfg["source"]["columns"]
    w = [f"{alias}.{sc['procedure_code']} IN ({basket_codes_sql(basket)})",
         f"{alias}.{sc['service_date']} <= '{cfg['end_date']}'"]
    if cfg["start_date"]:
        w.append(f"{alias}.{sc['service_date']} >= '{cfg['start_date']}'")
    return "\n  AND ".join(w)

def build_px_query_config(basket, cfg):
    s, sc = cfg["source"], cfg["source"]["columns"]
    return f"""
SELECT A.{sc['patient_id']} AS PATIENT_ID,
       MIN(A.{sc['service_date']}) AS FIRST_DIALYSIS_DATE,
       1 AS DIALYSIS_FLAG
FROM {s['table']} A
WHERE {_px_where(basket, cfg)}
GROUP BY 1
"""

px_query_config = build_px_query_config(px_basket, PX_CONFIG)
print(px_query_config)

In [0]:
# 3c. Run the config query
df_px_config = Get_Data_Snowflakes(px_query_config)
print(f"Config query: {df_px_config.count():,} patients (Step 2: {df_px_basket.count():,})")

In [0]:
# 3d. Compare Step 2 (basket) with Step 3 (basket + config). Expected: identical, except patients whose first
# date was after date_end (future-dated or after the run window).
print("Part A - all US patients, first dialysis date: Step 2 vs Step 3")
display(compare_first_dates(px_query_basket, px_query_config))
_late = Get_Data_Snowflakes(f"""
SELECT COUNT(*) AS ROWS_AFTER_END_DATE, COUNT(DISTINCT PATIENT_ID) AS PATIENTS, MAX(SERVICE_DATE) AS MAX_DATE
FROM {PX_CONFIG['source']['table']}
WHERE PROCEDURE_CODE IN ({basket_codes_sql(px_basket)}) AND SERVICE_DATE > '{PX_CONFIG['end_date']}'""").toPandas()
print(f"Dialysis rows after end_date {PX_CONFIG['end_date']}:")
display(_late)
print("Part C - validation patients, DIALYSIS_FLAG per patient-month")
display(compare_validation(df_px_basket, df_px_config))

### 3e. Findings: Step 3 vs Step 2 (run 2026-10-29)

| Check | Result |
| --- | --- |
| `PX_CONFIG` checks | Tables and columns found; all history → 2026-09-30 (`date_end`) |
| Same first date | 1,536,052 patients |
| Lost | 281 patients |
| Dialysis rows after `date_end` | 10,169 rows, 7,538 patients, latest 2026-10-02 |
| Validation patients (`DIALYSIS_FLAG` per month) | Identical: 8,413 months flagged, 363 patients, 0 changes |

**Observations**
* The only effect of the config is the end date. The 281 lost patients have their first dialysis on 2026-10-01 or 2026-10-02, after the run window. The other 7,257 patients with late rows keep their earlier first date.
* There are no future-dated rows (latest 2026-10-02, data refreshed at the start of October). The end date keeps the output aligned with the run window of the other journeys.
* Table and column names now come from `PX_CONFIG`. The start date stays empty on purpose (D3).

## Step 4: Output columns and dropped-rows report (Q7-Q9)

* One row per patient with `FIRST_DIALYSIS_DATE`, plus `FIRST_DIALYSIS_CODE` and `FIRST_DIALYSIS_GROUP` (code and `PX_GROUP` of the first event, ties: lowest code) for QC. The constant `DIALYSIS_FLAG = 1` is dropped: `06_patient_journey_combine` builds the flag itself and only reads `PATIENT_ID` and `FIRST_DIALYSIS_DATE`.
* Rows with an empty `PATIENT_ID` or `SERVICE_DATE` are dropped (3.0 grouped all empty patient IDs into one output row).
* A dropped-rows report counts every source row with a dialysis code under the first rule it fails, so losses are visible on every run.

| Cell | What it does |
| --- | --- |
| 4a | **The Px query** (Step 3 + output columns) and the dropped-rows report SQL |
| 4b | Runs it |
| 4c | Compares with Step 3 and shows the dropped-rows report |
| 4d | Findings |

In [0]:
# 4a. Step 4 query = Step 3 query + first code and group in the output, no constant flag, no empty patient / date;
# plus the dropped-rows report.

def _basket_groups_values(basket):
    _a = basket[basket["ACTIVE_DIALYSIS"] == "Y"].sort_values("PROCEDURE_CODE")
    v = ",\n         ".join(f"('{r.PROCEDURE_CODE}', '{r.PX_GROUP}')" for r in _a.itertuples())
    return f"SELECT column1 AS PROCEDURE_CODE, column2 AS PX_GROUP FROM VALUES\n         {v}"

def build_px_query_step4(basket, cfg):
    s, sc = cfg["source"], cfg["source"]["columns"]
    return f"""
SELECT A.{sc['patient_id']} AS PATIENT_ID,
       A.{sc['service_date']} AS FIRST_DIALYSIS_DATE,
       A.{sc['procedure_code']} AS FIRST_DIALYSIS_CODE,
       BSK.PX_GROUP AS FIRST_DIALYSIS_GROUP
FROM {s['table']} A
JOIN ({_basket_groups_values(basket)}
) BSK ON A.{sc['procedure_code']} = BSK.PROCEDURE_CODE
WHERE {_px_where(basket, cfg)}
  AND A.{sc['patient_id']} IS NOT NULL
  AND A.{sc['service_date']} IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY A.{sc['patient_id']} ORDER BY A.{sc['service_date']}, A.{sc['procedure_code']}) = 1
"""

def build_px_drop_report(basket, cfg):
    # every source row with a dialysis code (ACTIVE_DIALYSIS = Y), under the first rule it fails
    s, sc = cfg["source"], cfg["source"]["columns"]
    d, p = f"A.{sc['service_date']}", f"A.{sc['patient_id']}"
    start_rule = f"WHEN {d} < '{cfg['start_date']}' THEN '4. before start_date'" if cfg["start_date"] else ""
    return f"""
SELECT REASON, COUNT(*) AS ROWS_, COUNT(DISTINCT PATIENT_ID) AS PATIENTS
FROM (
  SELECT CASE WHEN {p} IS NULL THEN '1. no patient id'
              WHEN {d} IS NULL THEN '2. no service date'
              WHEN {d} > '{cfg['end_date']}' THEN '3. after end_date'
              {start_rule}
              ELSE 'kept' END AS REASON,
         {p} AS PATIENT_ID
  FROM {s['table']} A
  WHERE A.{sc['procedure_code']} IN ({basket_codes_sql(basket)})
)
GROUP BY 1 ORDER BY 1
"""

px_query_step4 = build_px_query_step4(px_basket, PX_CONFIG)
px_drop_report_sql = build_px_drop_report(px_basket, PX_CONFIG)
print(px_query_step4)

In [0]:
# 4b. Run the Step 4 query
df_px_step4 = Get_Data_Snowflakes(px_query_step4)
print(f"Step 4 query: {df_px_step4.count():,} patients (Step 3: {df_px_config.count():,})")
pd.DataFrame(df_px_step4.columns).display()

In [0]:
# 4c. Compare Step 3 with Step 4 (expected: identical first dates; only the empty-PATIENT_ID row goes),
# then the dropped-rows report and the first-event group per patient.
print("Part A - all US patients, first dialysis date: Step 3 vs Step 4")
display(compare_first_dates(px_query_config, px_query_step4))
print("Part C - validation patients, DIALYSIS_FLAG per patient-month")
display(compare_validation(df_px_config, df_px_step4))

print("Dropped-rows report (source rows with a dialysis code)")
_rep = Get_Data_Snowflakes(px_drop_report_sql).toPandas()
_rep["ROWS_"] = _rep["ROWS_"].astype("int64")
_rep["SHARE_ROWS_PCT"] = (100 * _rep["ROWS_"] / _rep["ROWS_"].sum()).round(4)   # in %, the drops are small
display(_rep)

print("First dialysis event by group (all US patients)")
display(df_px_step4.groupBy("FIRST_DIALYSIS_GROUP").agg(F.count("*").alias("PATIENTS")).orderBy(F.desc("PATIENTS")))

# Codes counted as dialysis that have no claims at all (candidates to check in the basket)
_codes = Get_Data_Snowflakes(f"""
SELECT PROCEDURE_CODE, COUNT(*) AS ROWS_ FROM {PX_CONFIG['source']['table']}
WHERE PROCEDURE_CODE IN ({basket_codes_sql(px_basket)}) GROUP BY 1""").toPandas()
_none = sorted(set(px_basket.loc[px_basket.ACTIVE_DIALYSIS == "Y", "PROCEDURE_CODE"]) - set(_codes.PROCEDURE_CODE))
print(f"Dialysis codes in the basket with no claims: {_none if _none else 'none'}")

### 4d. Findings: Step 4 vs Step 3 (run 2026-10-29)

| Check | Result |
| --- | --- |
| Patients | 1,536,052 (Step 3: 1,536,052), all with the same first date |
| Validation patients | Identical: 8,413 months flagged, 363 patients |
| Output columns | `PATIENT_ID`, `FIRST_DIALYSIS_DATE`, `FIRST_DIALYSIS_CODE`, `FIRST_DIALYSIS_GROUP` (constant `DIALYSIS_FLAG` dropped) |
| Dropped rows | After `date_end`: 10,169 rows (0.0044%); empty patient or date: 0; kept 230,410,450 rows |
| Dialysis codes with no claims | None (all 29 codes are used) |

**First dialysis event by group (all US):**

| Group | Patients | Share |
| --- | --- | --- |
| Dialysis session | 944,618 | 61.5% |
| ESRD monthly service | 525,030 | 34.2% |
| ESRD drug | 40,304 | 2.6% |
| Dialysis training | 26,100 | 1.7% |

**Observations**
* The code changes do not change any result. They only add QC columns and make the losses visible.
* `06_patient_journey_combine` reads only `PATIENT_ID` and `FIRST_DIALYSIS_DATE`, and builds `DIALYSIS_FLAG` itself, so dropping the constant flag is safe. **For review:** it drops only `FIRST_DIALYSIS_DATE` after the join, so the two QC columns would be carried into the combined table. Either drop them there, or write only the two columns used.

## Step 5: Write the output (after review)

Switched off (`WRITE_OUTPUT = False`) until the results above are reviewed. When switched on, it writes the Step 4 output to the same path as 3.0 (`kerendia_nba_3_1_px_validation`), which `06_patient_journey_combine` reads. Columns: `PATIENT_ID`, `FIRST_DIALYSIS_DATE` (used by PJ06), `FIRST_DIALYSIS_CODE`, `FIRST_DIALYSIS_GROUP` (QC). `DIALYSIS_FLAG` is no longer written: PJ06 builds it per month.

In [0]:
# Step 5. Write the Step 4 output (VALIDATION path, not production).
# The 3.1 Px table goes live only together with the 3.1 Dx table (decision D1, 2026-10-29), because the
# stage-5 exclusion of access-only patients comes from the 3.1 Dx output. End-to-end run 2026-10-30: all 3.1
# journeys write to their own dated folder (RUN_OUTPUT_DIR); the *_validation table PJ06 reads today is not touched.
WRITE_OUTPUT = True
RUN_OUTPUT_DIR = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                  "hcp_level_model/ops_pipeline/query_output/validation_2026_10_30/")
PX_OUTPUT_PATH = RUN_OUTPUT_DIR + "kerendia_nba_3_1_px_validation"
PX_OUTPUT_COLS = ["PATIENT_ID", "FIRST_DIALYSIS_DATE", "FIRST_DIALYSIS_CODE", "FIRST_DIALYSIS_GROUP"]

if WRITE_OUTPUT:
    (df_px_step4.select(*PX_OUTPUT_COLS)
     .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(PX_OUTPUT_PATH))
    _w = spark.read.format("delta").load(PX_OUTPUT_PATH)
    print(f"Written: {PX_OUTPUT_PATH} | {_w.count():,} patients, columns {_w.columns}")
else:
    print("WRITE_OUTPUT = False: nothing written (goes live with the 3.1 Dx table). "
          "The table still holds the 3.0 output of 2026-10-26.")